In [2]:
import os
import glob
import re
import random
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, classification_report

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

possible_paths = [
    "/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2",
    "/kaggle/input/celebdf-v2image-dataset/Celeb_V2"
]
BASE_DIR = next((p for p in possible_paths if os.path.exists(p)), possible_paths[0])
subfolders = os.listdir(BASE_DIR)
TRAIN_DIR = os.path.join(BASE_DIR, next(f for f in subfolders if f.lower() == 'train'))
VAL_DIR = os.path.join(BASE_DIR, next(f for f in subfolders if f.lower() == 'val'))

print(f"Train directory: {TRAIN_DIR}")
print(f"Validation directory: {VAL_DIR}")

Using device: cuda
Train directory: /kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2/Train
Validation directory: /kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2/Val


In [3]:
train_real = glob.glob(f"{TRAIN_DIR}/*real*/*.*") + glob.glob(f"{TRAIN_DIR}/*Real*/*.*")
train_fake = glob.glob(f"{TRAIN_DIR}/*fake*/*.*") + glob.glob(f"{TRAIN_DIR}/*Fake*/*.*")
val_real   = glob.glob(f"{VAL_DIR}/*real*/*.*")   + glob.glob(f"{VAL_DIR}/*Real*/*.*")
val_fake   = glob.glob(f"{VAL_DIR}/*fake*/*.*")   + glob.glob(f"{VAL_DIR}/*Fake*/*.*")

print(f"Original Train Real: {len(train_real):,} | Train Fake: {len(train_fake):,}")
print(f"Original Val Real:   {len(val_real):,} | Val Fake:   {len(val_fake):,}")

Original Train Real: 40,288 | Train Fake: 40,536
Original Val Real:   5,036 | Val Fake:   5,068


In [4]:
def get_video_id(path):
    fname = path.split('/')[-1]
    return fname.split('_face_')[0] if '_face_' in fname else None

all_real = train_real + val_real
all_fake = train_fake + val_fake

records = [(p, 0, get_video_id(p)) for p in all_real if get_video_id(p) is not None] + \
          [(p, 1, get_video_id(p)) for p in all_fake if get_video_id(p) is not None]

paths  = np.array([r[0] for r in records])
labels = np.array([r[1] for r in records])
groups = np.array([r[2] for r in records])

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(paths, labels, groups))

new_train_paths, new_train_labels = paths[train_idx], labels[train_idx]
new_val_paths, new_val_labels     = paths[val_idx], labels[val_idx]

overlap_check = set(groups[train_idx]) & set(groups[val_idx])
print(f"New Train: {len(new_train_paths)} images | New Val: {len(new_val_paths)} images")
print(f"Overlap: {len(overlap_check)} (must be 0)")
assert len(overlap_check) == 0

New Train: 77115 images | New Val: 13813 images
Overlap: 0 (must be 0)


In [7]:
class PathImageDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])


def get_balanced_capped_paths(paths, labels, max_per_class=10000, max_per_video=25, seed=42):
    paths = np.array(paths)
    labels = np.array(labels)
    video_ids = np.array([get_video_id(p) for p in paths])
    rng = np.random.RandomState(seed)

    capped_indices = []
    for vid in np.unique(video_ids):
        idx_pool = np.where(video_ids == vid)[0]
        chosen = rng.choice(idx_pool, size=min(max_per_video, len(idx_pool)), replace=False)
        capped_indices.extend(chosen)
    capped_indices = np.array(capped_indices)

    selected_indices = []
    for class_idx in np.unique(labels[capped_indices]):
        pool = capped_indices[labels[capped_indices] == class_idx]
        chosen = rng.choice(pool, size=min(max_per_class, len(pool)), replace=False)
        selected_indices.extend(chosen)

    rng.shuffle(selected_indices)
    selected_indices = np.array(selected_indices)
    return paths[selected_indices], labels[selected_indices]


sampled_train_paths, sampled_train_labels = get_balanced_capped_paths(
    new_train_paths, new_train_labels, max_per_class=10000, max_per_video=25
)
sampled_val_paths, sampled_val_labels = get_balanced_capped_paths(
    new_val_paths, new_val_labels, max_per_class=2000, max_per_video=25
)

print(f"Sampled Training set: {len(sampled_train_paths)} images")
print(f"Sampled Validation set: {len(sampled_val_paths)} images")

Sampled Training set: 20000 images
Sampled Validation set: 4000 images


In [8]:
class CustomDeepfakeCNN(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super(CustomDeepfakeCNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)
        self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(512 * 7 * 7, 512)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(512, 2)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.pool4(F.relu(self.bn4(self.conv4(x))))
        x = self.pool5(F.relu(self.bn5(self.conv5(x))))

        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [9]:
best = {
    "optimizer": "AdamW",
    "weight_decay": 3.869114342841159e-05,
    "dropout_rate": 0.470262086284662,
    "batch_size": 32,
    "lr": 0.00014417475659218634,
    "color_jitter": 0.3637075765616502,
    "blur_prob": 0.34448428673861303,
}
print("Using best hyperparameters:", best)

SAVE_PATH = "/kaggle/working/custom_cnn_deepfake_detector_final.pth"

final_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=best["color_jitter"], contrast=best["color_jitter"] * 0.7),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=5, sigma=(0.5, 2.0))], p=best["blur_prob"]),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = PathImageDataset(sampled_train_paths, sampled_train_labels, transform=final_train_transform)
val_dataset = PathImageDataset(sampled_val_paths, sampled_val_labels, transform=val_transform)

final_train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
final_val_loader = DataLoader(val_dataset, batch_size=best["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

final_model = CustomDeepfakeCNN(dropout_rate=best["dropout_rate"]).to(DEVICE)
optimizer = optim.AdamW(final_model.parameters(), lr=best["lr"], weight_decay=best["weight_decay"])
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda')

FINAL_EPOCHS = 30
PATIENCE = 5

best_val_f1 = 0.0
best_model_state = None
epochs_without_improvement = 0
history = {"train_loss": [], "val_f1": [], "val_auc": []}

print("\nStarting Final Training...")
for epoch in range(FINAL_EPOCHS):
    final_model.train()
    total_loss = 0.0
    for images, labels_ in final_train_loader:
        images, labels_ = images.to(DEVICE, non_blocking=True), labels_.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = final_model(images)
            loss = criterion(outputs, labels_)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(final_train_loader)

    final_model.eval()
    val_preds, val_probs, val_targets = [], [], []
    with torch.no_grad():
        for images, labels_ in final_val_loader:
            images = images.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda'):
                outputs = final_model(images)
                probs = torch.softmax(outputs, dim=1)[:, 1]
                preds = torch.argmax(outputs, dim=1)
            val_probs.extend(probs.cpu().numpy())
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(labels_.numpy())

    val_f1 = f1_score(val_targets, val_preds, average="macro")
    val_auc = roc_auc_score(val_targets, val_probs)

    history["train_loss"].append(avg_train_loss)
    history["val_f1"].append(val_f1)
    history["val_auc"].append(val_auc)

    print(f"Epoch [{epoch+1}/{FINAL_EPOCHS}] - Train Loss: {avg_train_loss:.4f} | Val F1: {val_f1:.4f} | Val AUC: {val_auc:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = copy.deepcopy(final_model.state_dict())
        epochs_without_improvement = 0

        # Turant disk pe save — session disconnect se bachne ke liye
        torch.save({
            'model_state_dict': best_model_state,
            'best_hyperparameters': best,
            'val_macro_f1': val_f1,
            'val_auc': val_auc,
            'epoch': epoch + 1,
            'training_history': history
        }, SAVE_PATH)
        print(f"  ↳ New best model (Val F1: {val_f1:.4f}) — SAVED to disk.")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

final_model.load_state_dict(best_model_state)
print(f"\nLoaded best model with Val Macro F1: {best_val_f1:.4f}")

Using best hyperparameters: {'optimizer': 'AdamW', 'weight_decay': 3.869114342841159e-05, 'dropout_rate': 0.470262086284662, 'batch_size': 32, 'lr': 0.00014417475659218634, 'color_jitter': 0.3637075765616502, 'blur_prob': 0.34448428673861303}

Starting Final Training...
Epoch [1/30] - Train Loss: 0.7045 | Val F1: 0.6871 | Val AUC: 0.7879
  ↳ New best model (Val F1: 0.6871) — SAVED to disk.
Epoch [2/30] - Train Loss: 0.5627 | Val F1: 0.7709 | Val AUC: 0.8492
  ↳ New best model (Val F1: 0.7709) — SAVED to disk.
Epoch [3/30] - Train Loss: 0.4813 | Val F1: 0.8210 | Val AUC: 0.8940
  ↳ New best model (Val F1: 0.8210) — SAVED to disk.
Epoch [4/30] - Train Loss: 0.4163 | Val F1: 0.8041 | Val AUC: 0.9145
Epoch [5/30] - Train Loss: 0.3653 | Val F1: 0.8397 | Val AUC: 0.9238
  ↳ New best model (Val F1: 0.8397) — SAVED to disk.
Epoch [6/30] - Train Loss: 0.3256 | Val F1: 0.8702 | Val AUC: 0.9454
  ↳ New best model (Val F1: 0.8702) — SAVED to disk.
Epoch [7/30] - Train Loss: 0.2879 | Val F1: 0.8723

In [10]:
final_model.eval()
all_preds, all_probs, all_targets = [], [], []
with torch.no_grad():
    for images, labels_ in final_val_loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda'):
            outputs = final_model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels_.numpy())

final_f1 = f1_score(all_targets, all_preds, average="macro")
final_auc = roc_auc_score(all_targets, all_probs)

print("--- Final Model Metrics ---")
print(f"Validation Macro F1: {final_f1:.4f}")
print(f"Validation AUC-ROC:  {final_auc:.4f}")
print(classification_report(all_targets, all_preds, target_names=['Authentic (0)', 'Manipulated (1)']))

size_mb = os.path.getsize(SAVE_PATH) / (1024 * 1024)
print(f"\n✅ Confirmed on disk: {SAVE_PATH} ({size_mb:.2f} MB)")
print("🚨 Ab turant 'Save Version -> Save & Run All (Commit)' kar dena, session pe chhodo mat.")

--- Final Model Metrics ---
Validation Macro F1: 0.9122
Validation AUC-ROC:  0.9721
                 precision    recall  f1-score   support

  Authentic (0)       0.91      0.91      0.91      2000
Manipulated (1)       0.91      0.91      0.91      2000

       accuracy                           0.91      4000
      macro avg       0.91      0.91      0.91      4000
   weighted avg       0.91      0.91      0.91      4000


✅ Confirmed on disk: /kaggle/working/custom_cnn_deepfake_detector_final.pth (55.02 MB)
🚨 Ab turant 'Save Version -> Save & Run All (Commit)' kar dena, session pe chhodo mat.
